# ` Intelligent E-Commerce Customer Analytics Platform `

# Step 1: Import Libraries

In [6]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

from scipy.sparse import csr_matrix

# Step 2: Load DataSet

In [7]:
file_path = r"C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed\customer_analytics_cleaned.csv"
recommendation_df = pd.read_csv(file_path)
print(recommendation_df.shape)
recommendation_df.head()

(119143, 45)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,4.0,No Title,No Comment,2017-05-26 00:00:00,2017-05-30 22:34:40,14409.0,-20.498489,-47.396929,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,5.0,No Title,No Comment,2018-01-30 00:00:00,2018-02-10 22:43:29,9790.0,-23.727992,-46.542848,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,5.0,No Title,No Comment,2018-06-15 00:00:00,2018-06-15 12:10:59,1151.0,-23.531642,-46.656289,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,5.0,No Title,No Comment,2018-03-29 00:00:00,2018-04-02 18:36:47,8775.0,-23.499702,-46.185233,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,5.0,a melhor nota,O baratheon è esxelente Amo adoro o baratheon,2018-08-10 00:00:00,2018-08-17 01:59:52,13056.0,-22.975100,-47.142925,campinas,SP


In [8]:
# Select Required Columns 

recommendation_df = recommendation_df[
    [
        "customer_unique_id",
        "product_id",
        "product_category_name_english"
    ]
]
recommendation_df.head()

,customer_unique_id,product_id,product_category_name_english
0,861eff4711a542e4b93843c6dd7febb0,a9516a079e37a9c9c36b9b78b10169e8,office_furniture
1,290c77bc529b7ac935b93aa66c333dc3,4aa6014eceb682077f9dc4bffebc05b0,housewares
2,060e732b5b29e8181a18229c7b0b2b5e,bd07b66896d6f1494f5b86251848ced7,office_furniture
3,259dac757896d24d7702b9acbbff3f3c,a5647c44af977b148e0a3a4751a09e2e,office_furniture
4,345ecd01c38d18a9036ed96c73b8d066,9391a573abe00141c56e38d84d7d5b3b,home_confort


In [9]:
# Handle Missing Values

recommendation_df.isnull().sum()

recommendation_df.dropna(inplace=True)
recommendation_df.reset_index(drop=True, inplace=True)

# Step 3 - Dataset Information

In [10]:
print("Shape:", recommendation_df.shape)
print("\nUnique Customers:", recommendation_df["customer_unique_id"].nunique())
print("Unique Products:",recommendation_df["product_id"].nunique())
print("\nTop Categories")
display(
    recommendation_df[
        "product_category_name_english"
    ].value_counts().head(10)
)

Shape: (118310, 3)

Unique Customers: 95420
Unique Products: 32951

Top Categories


product_category_name_english
bed_bath_table           11988
health_beauty            10032
sports_leisure            9004
furniture_decor           8832
computers_accessories     8150
housewares                7380
watches_gifts             6213
telephony                 4726
garden_tools              4590
auto                      4400
Name: count, dtype: int64

# 1. Content-Based Recommendation

## Create Product Dataset

In [11]:
product_df = recommendation_df[
    ["product_id", "product_category_name_english"]
].drop_duplicates()

print("Unique Products:", product_df.shape)
product_df.head()

Unique Products: (32951, 2)


,product_id,product_category_name_english
0,a9516a079e37a9c9c36b9b78b10169e8,office_furniture
1,4aa6014eceb682077f9dc4bffebc05b0,housewares
2,bd07b66896d6f1494f5b86251848ced7,office_furniture
3,a5647c44af977b148e0a3a4751a09e2e,office_furniture
4,9391a573abe00141c56e38d84d7d5b3b,home_confort


In [12]:
# Fill Missing Categories
product_df["product_category_name_english"] = (
    product_df["product_category_name_english"]
    .fillna("Unknown")
)

# Step 4 : Create TF-IDF Matrix

In [13]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform( product_df["product_category_name_english"])
print(tfidf_matrix.shape)

(32951, 72)


# Step 5: Compute Cosine Similarity

In [14]:
cosine_sim = cosine_similarity(tfidf_matrix)
print(cosine_sim.shape)

product_indices = pd.Series(
    product_df.index,
    index=product_df["product_id"]
).drop_duplicates()

product_indices.head()

(32951, 32951)


product_id
a9516a079e37a9c9c36b9b78b10169e8    0
4aa6014eceb682077f9dc4bffebc05b0    1
bd07b66896d6f1494f5b86251848ced7    2
a5647c44af977b148e0a3a4751a09e2e    3
9391a573abe00141c56e38d84d7d5b3b    4
dtype: int64

In [15]:
# Recommendation Function

def recommend_products(product_id, top_n=5):
    # Check whether product exists
    if product_id not in product_indices:
        return "Product ID not found."
    idx = product_indices[product_id]
    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]
    product_indices_list = [i[0] for i in similarity_scores ]
    return product_df.iloc[product_indices_list][ ["product_id","product_category_name_english"]]

In [16]:
# Testing the recommendation Function

sample_product = product_df["product_id"].iloc[0]
print("Selected Product:", sample_product)
recommend_products(sample_product)

Selected Product: a9516a079e37a9c9c36b9b78b10169e8


,product_id,product_category_name_english
2,bd07b66896d6f1494f5b86251848ced7,office_furniture
3,a5647c44af977b148e0a3a4751a09e2e,office_furniture
43,8ed094bfe076c568f6bb10feada3f75d,office_furniture
74,aba86c093ccdbac75b09111d57e50004,office_furniture
302,eab67bf937aaadc19f83383a331d2dd9,office_furniture


# Step 6: Interpretation

    - This recommendation engine identifies products with similar categories. 
    - For example, if a customer views or purchases a product in the office_furniture category, the system recommends other products from the same category.

# 2. Collaborative Filtering

## Step 1 – Create Customer–Product Matrix

In [17]:
# Create customer-product interactions
customer_product = recommendation_df[
    ["customer_unique_id", "product_id"]
].drop_duplicates()

# Convert IDs to category codes
customer_product["customer_code"] = (
    customer_product["customer_unique_id"]
    .astype("category")
    .cat.codes
)

customer_product["product_code"] = (
    customer_product["product_id"]
    .astype("category")
    .cat.codes
)

# Create sparse matrix
customer_product_sparse = csr_matrix(
    (
        np.ones(len(customer_product)),
        (
            customer_product["customer_code"],
            customer_product["product_code"]
        )
    )
)

print("customer product sparse shape: ",customer_product_sparse.shape)

customer_similarity = cosine_similarity(
    customer_product_sparse[:5000]
)

top_customers = (
    recommendation_df["customer_unique_id"]
    .value_counts()
    .head(5000)
    .index
)

recommendation_small = recommendation_df[
    recommendation_df["customer_unique_id"]
    .isin(top_customers)
]
customer_product_matrix = pd.crosstab(
    recommendation_small["customer_unique_id"],
    recommendation_small["product_id"]
)

customer_product_matrix = (
    customer_product_matrix > 0
).astype(np.int8)

print("customer product matrix shape: ",customer_product_matrix.shape)

customer product sparse shape:  (95420, 32951)
customer product matrix shape:  (5000, 5958)


In [18]:
# Compute Customer Similarity

customer_similarity = cosine_similarity(customer_product_matrix)

customer_similarity_df = pd.DataFrame(
    customer_similarity,
    index=customer_product_matrix.index,
    columns=customer_product_matrix.index
)

print(customer_similarity_df.shape)
# customer_similarity_df.iloc[:5, :5]

(5000, 5000)


# Step 2: Recommendation Function

In [19]:
def recommend_for_customer(customer_id, top_n=5):

    if customer_id not in customer_similarity_df.index:
        return "Customer not found."

    # Find top 10 similar customers
    similar_customers = (
        customer_similarity_df[customer_id]
        .sort_values(ascending=False)
        .iloc[1:11]
        .index
    )

    # Products bought by similar customers
    similar_products = customer_product_matrix.loc[similar_customers].sum()

    # Products already bought by target customer
    purchased_products = customer_product_matrix.loc[customer_id]

    recommendations = (
        similar_products[purchased_products == 0]
        .sort_values(ascending=False)
        .head(top_n)
    )

    return recommendations

# Step 3: Test The Model

In [20]:
sample_customer = customer_product_matrix.index[0]
print("Customer:", sample_customer)
recommendations = recommend_for_customer(sample_customer)
print(recommendations)

Customer: 000bfa1d2f1a41876493be685390d6d3
product_id
43ee88561093499d9e571d4db5f20b79    1
4630761de87581e8b659dc77bb7eb4ee    1
16679d68bdbed3fb97af806424d7f2b7    1
7c6ae4d12d2754ca0826ad17792c554e    1
368c6c730842d78016ad823897a372db    1
dtype: int64


In [21]:

recommended_ids = recommendations.index.tolist()

recommended_products = (
    recommendation_df[
        recommendation_df["product_id"].isin(recommended_ids)
    ][
        ["product_id", "product_category_name_english"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

recommended_products

,product_id,product_category_name_english
0,368c6c730842d78016ad823897a372db,garden_tools
1,16679d68bdbed3fb97af806424d7f2b7,health_beauty
2,43ee88561093499d9e571d4db5f20b79,furniture_decor
3,7c6ae4d12d2754ca0826ad17792c554e,sports_leisure
4,4630761de87581e8b659dc77bb7eb4ee,luggage_accessories


# Step 4: Interpretation

    - Based on the products purchased by these similar customers, the system recommended five products that the target customer has not previously purchased. 
    - The recommended products span multiple categories, including Garden Tools, Health & Beauty, Furniture & Decor, Sports & Leisure, and Luggage & Accessories, enabling opportunities for cross-selling and increasing customer engagement.

In [ ]:
import joblib

joblib.dump(
    customer_similarity_df,
    "../artifacts/customer_similarity.pkl"
)

joblib.dump(
    customer_product_matrix,
    "../artifacts/customer_product_matrix.pkl"
)
print("Recommendation model saved successfully.")


Recommendation model saved successfully.
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>


In [ ]:
test_customer = "003e2af81ea3c72fbdd093f07724a4de"

similarities = customer_similarity_df.loc[test_customer]

print("Customer:", test_customer)


print("\nNumber of positive similarities:")
print((similarities > 0).sum())

print("\nTop similarities:")
print(
    similarities[
        similarities > 0
    ]
    .sort_values(ascending=False)
    .head(10)
)

print("Purchased products:")

purchased = customer_product_matrix.loc[test_customer]

purchased = purchased[
    purchased > 0
]

print(purchased)

recommendations = recommend_products(
    customer_id=test_customer,
    customer_similarity_df=customer_similarity_df,
    customer_product_matrix=customer_product_matrix,
    top_n=5
)

print("Recommendations:")
print(recommendations)

Customer: 003e2af81ea3c72fbdd093f07724a4de

Number of positive similarities:
7

Top similarities:
customer_unique_id
003e2af81ea3c72fbdd093f07724a4de    1.0
38244f285d64696e19b5d2ff854a3ea3    1.0
755b01b2c86cb92e6b25d45f9fa8c238    1.0
9648ed501d8dbf31fda943cd91da7d18    1.0
bf06b935e76a7545da01410a6befbb1c    1.0
cfa9a212b3e5986d1b03abe4c2acff4b    1.0
d1ac08467e5ac5e907b719c89c80ff2c    1.0
Name: 003e2af81ea3c72fbdd093f07724a4de, dtype: float64
Purchased products:
product_id
16679d68bdbed3fb97af806424d7f2b7    1
Name: 003e2af81ea3c72fbdd093f07724a4de, dtype: int8
Recommendations:
[]


In [47]:
def find_recommendable_customer(
    similarity_df,
    product_matrix
):

    for customer_id in similarity_df.index:

        similarities = similarity_df.loc[
            customer_id
        ].drop(customer_id)

        similar_customers = similarities[
            similarities > 0
        ].sort_values(
            ascending=False
        ).head(20)

        if similar_customers.empty:
            continue

        purchased = set(
            product_matrix.loc[customer_id][
                product_matrix.loc[customer_id] > 0
            ].index
        )

        similar_products = set()

        for similar_customer in similar_customers.index:

            products = product_matrix.loc[
                similar_customer
            ]

            products = products[
                products > 0
            ].index

            similar_products.update(products)

        new_products = (
            similar_products - purchased
        )

        if new_products:

            return customer_id, new_products

    return None, set()

In [49]:
test_customer, candidates = find_recommendable_customer(
    customer_similarity_df,
    customer_product_matrix
)

print("Test Customer:")
print(test_customer)

print("\nRecommendation Candidates:")
print(candidates)

Test Customer:
0058f300f57d7b93c477a131a59b36c3

Recommendation Candidates:
{'ca753b1e379d59868172ac9d8d62d14f', '819fc0e9c061a08bac3d5044eb92b173', '19c91ef95d509ea33eda93495c4d3481'}


In [50]:
test_customer = "0058f300f57d7b93c477a131a59b36c3"

recommendations = recommend_products(
    customer_id=test_customer,
    customer_similarity_df=customer_similarity_df,
    customer_product_matrix=customer_product_matrix,
    top_n=5
)

print("Recommendations:")
print(recommendations)

print("\nType:")
print(type(recommendations))

print("\nNumber of recommendations:")
print(len(recommendations))

Recommendations:
['19c91ef95d509ea33eda93495c4d3481', '819fc0e9c061a08bac3d5044eb92b173', 'ca753b1e379d59868172ac9d8d62d14f']

Type:
<class 'list'>

Number of recommendations:
3
